# Reviewed Attention U-Net final candidate

**Colab: Python 3, T4 GPU. Run cells in order from the top.**

Combination: saved Attention U-Net → validation-selected mask threshold and
single/four-flip prediction → RGB ROI with 15% padding + original image →
balanced-weight EfficientNetB0 → validation-selected classifier checkpoint,
single/four-flip prediction, and decision threshold.

Training uses flips, small rotations, contrast and modest brightness changes.
Classifier checkpoints maximise specificity while maintaining at least 80%
validation sensitivity, with AUC as a tie-breaker. Segmentation inference is
chosen by per-image mean validation Dice. These are separate selection rules:
better Dice does not guarantee better classification.

The split remains 720 training / 180 validation / 379 official-test images.
The official test was viewed earlier, so the final result is an exploratory
follow-up. Do not use it to change this run's settings or select a model.
This is a research prototype, not a clinically validated diagnosis system.


## 1. Imports

In [ ]:
import os
import gc
import json
import random
import shutil
import time
import zipfile
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


## 2. Settings and output folder

In [ ]:
SEED = 42
keras.utils.set_random_seed(SEED)
SEG_IMAGE_SIZE = (256, 256)
CLS_IMAGE_SIZE = (224, 224)
SEG_BATCH_SIZE = 4
CLS_BATCH_SIZE = 8
SEG_EPOCHS = 35
CLS_HEAD_EPOCHS = 15
CLS_FINE_TUNE_EPOCHS = 8
SEG_LEARNING_RATE = 1e-3
CLS_HEAD_LR = 3e-4
CLS_FINE_TUNE_LR = 1e-5
SEG_THRESHOLD = 0.5
TARGET_VALIDATION_SENSITIVITY = 0.80
SAVE_RESULTS_TO_DRIVE = True

RUNTIME_ROOT = Path('/content/melanoma_isic2016_final_candidate')
DOWNLOAD_DIR = RUNTIME_ROOT / 'downloads'
DATA_DIR = RUNTIME_ROOT / 'images'
ROI_DIR = RUNTIME_ROOT / 'predicted_roi'
PRED_MASK_DIR = RUNTIME_ROOT / 'predicted_masks'
for folder in (DOWNLOAD_DIR, DATA_DIR, ROI_DIR, PRED_MASK_DIR):
    folder.mkdir(parents=True, exist_ok=True)
if SAVE_RESULTS_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_DIR = Path('/content/drive/MyDrive/Melanoma_ISIC2016_Final_Candidate')
    except Exception as exc:
        print('Drive unavailable; writing output in /content:', exc)
        OUTPUT_DIR = RUNTIME_ROOT / 'results'
else:
    OUTPUT_DIR = RUNTIME_ROOT / 'results'
# Each execution gets its own folder, preserving earlier checkpoints.
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_DIR = OUTPUT_DIR / RUN_ID
MODELS_DIR = OUTPUT_DIR / 'models'
RESULTS_DIR = OUTPUT_DIR / 'tables'
for folder in (MODELS_DIR, RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print('Results:', OUTPUT_DIR)

# Reuse the saved segmenter from the most recent completed run to avoid
# spending time retraining a stage whose test Dice was already about 0.888.
PREVIOUS_SEGMENTER = Path(
    '/content/drive/MyDrive/Melanoma_ISIC2016_Improved_Results/models/attention_unet_best.keras'
)
REUSE_SAVED_SEGMENTER = True


## 3. Download the official ISIC 2016 Task 3B files

This may download about 850 MB. If Colab disconnects, downloaded runtime files
may be lost. The official test labels are held out from all training, model
selection, and threshold selection.

In [ ]:
BASE_URL = 'https://isic-archive.s3.amazonaws.com/challenges/2016/'
files = {
    'training_zip': 'ISBI2016_ISIC_Part3B_Training_Data.zip',
    'training_csv': 'ISBI2016_ISIC_Part3B_Training_GroundTruth.csv',
    'test_zip': 'ISBI2016_ISIC_Part3B_Test_Data.zip',
    'test_csv': 'ISBI2016_ISIC_Part3B_Test_GroundTruth.csv',
}

def download(url, destination):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 100:
        print('Already downloaded:', destination.name)
        return destination
    temporary = destination.with_suffix(destination.suffix + '.part')
    start = temporary.stat().st_size if temporary.exists() else 0
    headers = {'Range': f'bytes={start}-'} if start else {}
    print('Downloading:', destination.name)
    with requests.get(url, stream=True, headers=headers, timeout=(30, 120)) as response:
        response.raise_for_status()
        if start and response.status_code != 206:
            start = 0
            temporary.unlink(missing_ok=True)
        with open(temporary, 'ab' if start else 'wb') as handle:
            for part in response.iter_content(chunk_size=8 * 1024 * 1024):
                if part:
                    handle.write(part)
    temporary.replace(destination)
    return destination

paths = {key: download(BASE_URL + filename, DOWNLOAD_DIR / filename)
         for key, filename in files.items()}
for key in ('training_zip', 'test_zip'):
    assert zipfile.is_zipfile(paths[key]), f'Invalid ZIP: {paths[key]}'
print('Official files downloaded.')


## 4. Make a stratified 720/180 training/validation split; lock the official test set

In [ ]:
def read_labels(path, expected):
    # The 2016 training CSV uses benign/malignant, while the official test
    # CSV uses numeric 0/1. Per ISIC, 0 = benign and 1 = malignant.
    frame = pd.read_csv(path, header=None, names=['image_id', 'raw_label'],
                        dtype=str)
    frame['image_id'] = frame['image_id'].astype(str).str.strip()
    normalized = frame['raw_label'].astype(str).str.strip().str.lower()
    frame['label'] = normalized.map({
        'benign': 0, 'malignant': 1, '0': 0, '0.0': 0, '1': 1, '1.0': 1,
    })
    frame = frame[frame['image_id'].str.fullmatch(r'ISIC_\d{7}') &
                  frame['label'].notna()].copy()
    assert len(frame) == expected and frame['image_id'].is_unique, (
        f'Expected {expected} unique labels in {path}, got {len(frame)}. '
        'Check CSV columns and label values.'
    )
    frame['label'] = frame['label'].astype(int)
    frame['diagnosis'] = np.where(frame['label'] == 1, 'malignant', 'benign')
    return frame[['image_id', 'diagnosis', 'label']].reset_index(drop=True)

training_labels = read_labels(paths['training_csv'], 900)
official_test_labels = read_labels(paths['test_csv'], 379)
assert set(training_labels.image_id).isdisjoint(set(official_test_labels.image_id))
train_df, val_df = train_test_split(
    training_labels, test_size=0.20, stratify=training_labels['label'],
    random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = official_test_labels.copy()
for name, frame in [('train', train_df), ('validation', val_df), ('official_test', test_df)]:
    print(name, 'total:', len(frame), 'melanoma:', int(frame['label'].sum()))
    frame[['image_id', 'diagnosis', 'label']].to_csv(
        RESULTS_DIR / f'{name}_image_ids.csv', index=False)


## 5. Extract the matched images and expert masks

In [ ]:
def extract_pairs(zip_path, labels, split_name):
    destination = DATA_DIR / split_name
    destination.mkdir(parents=True, exist_ok=True)
    records = []
    with zipfile.ZipFile(zip_path) as archive:
        member_map = {Path(name).name: name for name in archive.namelist()
                      if not name.endswith('/')}
        for image_id in labels['image_id']:
            image_name = f'{image_id}.jpg'
            mask_name = f'{image_id}_Segmentation.png'
            assert image_name in member_map and mask_name in member_map, image_id
            image_path, mask_path = destination / image_name, destination / mask_name
            for name, out in [(image_name, image_path), (mask_name, mask_path)]:
                if not out.exists():
                    with archive.open(member_map[name]) as src, open(out, 'wb') as dst:
                        shutil.copyfileobj(src, dst)
            records.append({'image_id': image_id, 'image_path': str(image_path),
                            'mask_path': str(mask_path)})
    return pd.DataFrame(records)

training_paths = extract_pairs(paths['training_zip'], training_labels, 'development')
test_paths = extract_pairs(paths['test_zip'], test_df, 'official_test')
train_df = train_df.merge(training_paths, on='image_id', validate='one_to_one')
val_df = val_df.merge(training_paths, on='image_id', validate='one_to_one')
test_df = test_df.merge(test_paths, on='image_id', validate='one_to_one')
for key in ('training_zip', 'test_zip'):
    paths[key].unlink(missing_ok=True)
print('Extracted:', len(train_df), len(val_df), len(test_df))


## 6. Visual check: image, expert lesion mask, overlay

In [ ]:
def load_rgb(path):
    with Image.open(path) as picture:
        return picture.convert('RGB').copy()

def load_mask(path):
    with Image.open(path) as picture:
        return picture.convert('L').copy()

def segmentation_overlay(image_pil, mask_binary, alpha=0.35):
    image = np.asarray(image_pil.convert('RGB'), dtype=np.float32).copy()
    mask = Image.fromarray(np.uint8(np.asarray(mask_binary) > 0) * 255)
    mask = np.asarray(mask.resize(image_pil.size, Image.Resampling.NEAREST)) > 0
    image[mask] = (1-alpha) * image[mask] + alpha * np.array([255, 0, 0])
    return Image.fromarray(np.uint8(np.clip(image, 0, 255)))

examples = pd.concat([train_df.query('label == 0').head(2),
                      train_df.query('label == 1').head(2)])
fig, axes = plt.subplots(len(examples), 3, figsize=(12, 3.4*len(examples)))
for i, (_, row) in enumerate(examples.iterrows()):
    image = load_rgb(row['image_path'])
    mask = np.asarray(load_mask(row['mask_path'])) > 127
    for j, (panel, title) in enumerate(zip(
        [image, mask, segmentation_overlay(image, mask)],
        [f"{row['image_id']} {row['diagnosis']}", 'Expert mask', 'Overlay'])):
        axes[i, j].imshow(panel, cmap='gray' if j == 1 else None)
        axes[i, j].set_title(title)
        axes[i, j].axis('off')
plt.tight_layout(); plt.show()


## 7. Attention U-Net segmentation data, paired augmentation, Dice and IoU

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_segmentation_image(image_path):
    image = tf.io.decode_jpeg(tf.io.read_file(image_path), channels=3)
    return tf.cast(tf.image.resize(image, SEG_IMAGE_SIZE, method='bilinear'),
                   tf.float32) / 255.0

def decode_segmentation_mask(mask_path):
    mask = tf.io.decode_png(tf.io.read_file(mask_path), channels=1)
    mask = tf.image.resize(mask, SEG_IMAGE_SIZE, method='nearest')
    return tf.cast(mask > 127, tf.float32)

def decode_segmentation_pair(image_path, mask_path):
    return decode_segmentation_image(image_path), decode_segmentation_mask(mask_path)

def paired_segmentation_augmentation(image, mask):
    seed1 = tf.random.uniform([2], maxval=2**31-1, dtype=tf.int32)
    seed2 = tf.random.uniform([2], maxval=2**31-1, dtype=tf.int32)
    image = tf.image.stateless_random_flip_left_right(image, seed1)
    mask = tf.image.stateless_random_flip_left_right(mask, seed1)
    image = tf.image.stateless_random_flip_up_down(image, seed2)
    mask = tf.image.stateless_random_flip_up_down(mask, seed2)
    turns = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
    image, mask = tf.image.rot90(image, turns), tf.image.rot90(mask, turns)
    image = tf.image.random_brightness(image, max_delta=0.06)
    image = tf.image.random_contrast(image, lower=0.90, upper=1.10)
    return tf.clip_by_value(image, 0.0, 1.0), mask

def make_seg_dataset(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((
        frame['image_path'].astype(str).values,
        frame['mask_path'].astype(str).values))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_segmentation_pair, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(paired_segmentation_augmentation, num_parallel_calls=AUTOTUNE)
    return ds.batch(SEG_BATCH_SIZE).prefetch(AUTOTUNE)

train_seg_ds = make_seg_dataset(train_df, training=True)
val_seg_ds = make_seg_dataset(val_df)
test_seg_ds = make_seg_dataset(test_df)
images_batch, masks_batch = next(iter(train_seg_ds))
print('Image and mask shapes:', images_batch.shape, masks_batch.shape)
print('Image range:', float(tf.reduce_min(images_batch)),
      'to', float(tf.reduce_max(images_batch)))
print('Mask values:', np.unique(masks_batch.numpy()))


In [ ]:
EPS = 1e-7
SPATIAL_AXES = (1, 2, 3)

@keras.utils.register_keras_serializable(package='MelanomaAI')
def soft_dice_coefficient(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=SPATIAL_AXES)
    denominator = tf.reduce_sum(y_true + y_pred, axis=SPATIAL_AXES)
    return tf.reduce_mean((2.0 * intersection + EPS) / (denominator + EPS))

@keras.utils.register_keras_serializable(package='MelanomaAI')
def dice_coefficient(y_true, y_pred):
    # Hard per-image Dice averaged across the current batch.
    y_true = tf.cast(y_true > 0.5, tf.float32)
    y_pred = tf.cast(y_pred >= SEG_THRESHOLD, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=SPATIAL_AXES)
    denominator = tf.reduce_sum(y_true + y_pred, axis=SPATIAL_AXES)
    return tf.reduce_mean((2.0 * intersection + EPS) / (denominator + EPS))

@keras.utils.register_keras_serializable(package='MelanomaAI')
def dice_loss(y_true, y_pred):
    return 1.0 - soft_dice_coefficient(y_true, y_pred)

@keras.utils.register_keras_serializable(package='MelanomaAI')
def iou_coefficient(y_true, y_pred):
    y_true = tf.cast(y_true > 0.5, tf.float32)
    y_pred = tf.cast(y_pred >= SEG_THRESHOLD, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=SPATIAL_AXES)
    union = tf.reduce_sum(y_true + y_pred - y_true * y_pred, axis=SPATIAL_AXES)
    return tf.reduce_mean((intersection + EPS) / (union + EPS))

@keras.utils.register_keras_serializable(package='MelanomaAI')
def combined_segmentation_loss(y_true, y_pred):
    bce = tf.reduce_mean(keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + dice_loss(y_true, y_pred)


## 8. Attention U-Net architecture and training

In [ ]:
def conv_block(
    x,
    filters,
    name,
):
    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False,
        name=f"{name}_conv1",
    )(x)

    x = layers.BatchNormalization(
        name=f"{name}_bn1"
    )(x)

    x = layers.Activation(
        "relu",
        name=f"{name}_relu1",
    )(x)

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False,
        name=f"{name}_conv2",
    )(x)

    x = layers.BatchNormalization(
        name=f"{name}_bn2"
    )(x)

    x = layers.Activation(
        "relu",
        name=f"{name}_relu2",
    )(x)

    return x


def attention_gate(
    skip,
    gating,
    inter_filters,
    name,
):
    theta = layers.Conv2D(
        inter_filters,
        1,
        padding="same",
        name=f"{name}_theta",
    )(skip)

    phi = layers.Conv2D(
        inter_filters,
        1,
        padding="same",
        name=f"{name}_phi",
    )(gating)

    x = layers.Add(
        name=f"{name}_add"
    )([theta, phi])

    x = layers.Activation(
        "relu",
        name=f"{name}_relu",
    )(x)

    psi = layers.Conv2D(
        1,
        1,
        padding="same",
        activation="sigmoid",
        name=f"{name}_psi",
    )(x)

    return layers.Multiply(
        name=f"{name}_weighted_skip"
    )([skip, psi])


def build_attention_unet(
    input_shape=(256, 256, 3),
):
    inputs = keras.Input(
        shape=input_shape,
        name="image",
    )

    c1 = conv_block(inputs, 32, "enc1")
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 64, "enc2")
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 128, "enc3")
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 256, "enc4")
    p4 = layers.MaxPooling2D()(c4)

    bottleneck = conv_block(
        p4,
        512,
        "bottleneck",
    )

    def decoder_step(
        x,
        skip,
        filters,
        level,
    ):
        x = layers.Conv2DTranspose(
            filters,
            2,
            strides=2,
            padding="same",
            name=f"up{level}",
        )(x)

        skip = attention_gate(
            skip,
            x,
            max(filters // 2, 1),
            name=f"attention{level}",
        )

        x = layers.Concatenate(
            name=f"concat{level}"
        )([x, skip])

        return conv_block(
            x,
            filters,
            f"dec{level}",
        )

    d4 = decoder_step(bottleneck, c4, 256, 4)
    d3 = decoder_step(d4, c3, 128, 3)
    d2 = decoder_step(d3, c2, 64, 2)
    d1 = decoder_step(d2, c1, 32, 1)

    outputs = layers.Conv2D(
        1,
        1,
        activation="sigmoid",
        dtype="float32",
        name="segmentation_mask",
    )(d1)

    return keras.Model(
        inputs,
        outputs,
        name="Attention_U_Net",
    )


In [ ]:
CUSTOM_OBJECTS = {
    "dice_coefficient": dice_coefficient,
    "soft_dice_coefficient": soft_dice_coefficient,
    "dice_loss": dice_loss,
    "iou_coefficient": iou_coefficient,
    "combined_segmentation_loss": (
        combined_segmentation_loss
    ),
}


def compile_segmentation_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=SEG_LEARNING_RATE
        ),
        loss=combined_segmentation_loss,
        metrics=[
            dice_coefficient,
            iou_coefficient,
        ],
    )


def segmentation_callbacks(model_path):
    return [
        keras.callbacks.ModelCheckpoint(
            model_path,
            monitor="val_dice_coefficient",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_dice_coefficient",
            mode="max",
            patience=8,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_dice_coefficient",
            mode="max",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1,
        ),
    ]


ATTENTION_MODEL_PATH = MODELS_DIR / 'attention_unet_best.keras'

if REUSE_SAVED_SEGMENTER and PREVIOUS_SEGMENTER.is_file():
    shutil.copy2(PREVIOUS_SEGMENTER, ATTENTION_MODEL_PATH)
    attention_training_seconds = 0.0
    attention_history = {}
    print('Reusing saved Attention U-Net:', PREVIOUS_SEGMENTER)
else:
    print('Saved segmenter unavailable; training Attention U-Net.')
    tf.keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    attention_model = build_attention_unet(input_shape=(*SEG_IMAGE_SIZE, 3))
    compile_segmentation_model(attention_model)
    start_time = time.time()
    attention_fit = attention_model.fit(
        train_seg_ds, validation_data=val_seg_ds,
        epochs=SEG_EPOCHS,
        callbacks=segmentation_callbacks(ATTENTION_MODEL_PATH), verbose=1,
    )
    attention_training_seconds = time.time() - start_time
    attention_history = attention_fit.history
    del attention_model, attention_fit
    gc.collect()

selected_segmenter = keras.models.load_model(
    ATTENTION_MODEL_PATH, custom_objects=CUSTOM_OBJECTS)
attention_validation = selected_segmenter.evaluate(
    val_seg_ds, verbose=0, return_dict=True)
SELECTED_SEGMENTER_NAME = 'Attention U-Net'
SELECTED_SEGMENTER_PATH = str(ATTENTION_MODEL_PATH)
print('Validation Dice before inference selection (threshold 0.5):', attention_validation['dice_coefficient'])
print('Validation IoU before inference selection (threshold 0.5):', attention_validation['iou_coefficient'])
print('Segmentation parameters:', selected_segmenter.count_params())


def predict_segmentation(images, four_view=False, model=None):
    model = selected_segmenter if model is None else model
    images = np.asarray(images, dtype=np.float32)
    if not four_view:
        return model.predict(images, batch_size=SEG_BATCH_SIZE, verbose=0)[..., 0]
    predictions = []
    for axes in [(), (2,), (1,), (1, 2)]:
        transformed = np.flip(images, axis=axes) if axes else images
        scores = model.predict(np.ascontiguousarray(transformed),
                               batch_size=SEG_BATCH_SIZE, verbose=0)[..., 0]
        # Undo each flip before averaging the spatial probability maps.
        scores = np.flip(scores, axis=axes) if axes else scores
        predictions.append(scores)
    return np.mean(np.stack(predictions), axis=0)

def mask_metrics(truth, probabilities, threshold):
    prediction = probabilities >= threshold
    intersection = np.logical_and(truth, prediction).sum(axis=(1, 2))
    total = truth.sum(axis=(1, 2)) + prediction.sum(axis=(1, 2))
    union = np.logical_or(truth, prediction).sum(axis=(1, 2))
    return (2*intersection + 1e-7)/(total + 1e-7), (intersection + 1e-7)/(union + 1e-7)

segmentation_selection_rows = []
for method, use_flips in [('single', False), ('4-flip average', True)]:
    dice_parts = {t: [] for t in (0.4, 0.5, 0.6)}
    iou_parts = {t: [] for t in (0.4, 0.5, 0.6)}
    for start in range(0, len(val_df), 12):
        rows = val_df.iloc[start:start+12]
        images = np.stack([decode_segmentation_image(p).numpy()
                           for p in rows.image_path])
        truth = np.stack([decode_segmentation_mask(p).numpy()[..., 0] > 0.5
                          for p in rows.mask_path])
        scores = predict_segmentation(images, four_view=use_flips)
        for threshold in dice_parts:
            dice, iou = mask_metrics(truth, scores, threshold)
            dice_parts[threshold].extend(dice.tolist())
            iou_parts[threshold].extend(iou.tolist())
    for threshold in dice_parts:
        segmentation_selection_rows.append({
            'Method': method, 'Threshold': threshold,
            'Validation Dice': float(np.mean(dice_parts[threshold])),
            'Validation IoU': float(np.mean(iou_parts[threshold])),
            'Worst validation Dice': float(np.min(dice_parts[threshold])),
        })

segmentation_selection = pd.DataFrame(segmentation_selection_rows).sort_values(
    ['Validation Dice', 'Validation IoU'], ascending=False).reset_index(drop=True)
SELECTED_SEGMENTATION_METHOD = str(segmentation_selection.loc[0, 'Method'])
SEG_THRESHOLD = float(segmentation_selection.loc[0, 'Threshold'])
display(segmentation_selection.round(4))
print('Selected mask inference:', SELECTED_SEGMENTATION_METHOD, SEG_THRESHOLD)
segmentation_selection.to_csv(RESULTS_DIR / 'validation_segmentation_inference.csv',
                              index=False)
SEGMENTER_REUSED = bool(REUSE_SAVED_SEGMENTER and PREVIOUS_SEGMENTER.is_file())
checkpoint_hash = hashlib.sha256(ATTENTION_MODEL_PATH.read_bytes()).hexdigest()
split_hash = hashlib.sha256(('|'.join(sorted(train_df.image_id)) + '\n' +
                             '|'.join(sorted(val_df.image_id))).encode()).hexdigest()
(RESULTS_DIR / 'segmentation_configuration.json').write_text(json.dumps({
    'method': SELECTED_SEGMENTATION_METHOD, 'threshold': SEG_THRESHOLD,
    'checkpoint_sha256': checkpoint_hash, 'split_sha256': split_hash,
    'checkpoint_reused': SEGMENTER_REUSED,
}, indent=2))


## 9. Generate predicted masks and RGB lesion ROIs

Use the mask prediction method and threshold selected by validation Dice.
The crop preserves the lesion's shape with 15% padding and an RGB background.
Empty or implausible masks fall back to the original image. Training ROIs are
generated by the segmenter trained on those training images; this is a limitation
to report, while validation/test ROIs are produced without their expert masks.


In [ ]:
from scipy import ndimage

def crop_from_prediction(image_pil, predicted_mask, padding=0.15):
    image = image_pil.convert('RGB')
    width, height = image.size
    binary = Image.fromarray(np.uint8(np.asarray(predicted_mask) > 0) * 255)
    binary = np.asarray(binary.resize(image.size, Image.Resampling.NEAREST)) > 0
    labelled, count = ndimage.label(binary)
    reason = 'cropped'
    if count == 0:
        crop, reason = image, 'empty predicted mask'
    else:
        sizes = np.bincount(labelled.ravel())
        sizes[0] = 0
        lesion = labelled == int(sizes.argmax())
        area = int(lesion.sum())
        if area < max(64, int(width*height*0.001)) or area > width*height*0.80:
            crop, reason = image, 'implausible predicted area'
        else:
            ys, xs = np.where(lesion)
            span = max(np.ptp(xs)+1, np.ptp(ys)+1)
            margin = int(np.ceil(span*padding))
            x0, x1 = max(0, int(xs.min())-margin), min(width, int(xs.max())+margin+1)
            y0, y1 = max(0, int(ys.min())-margin), min(height, int(ys.max())+margin+1)
            crop = image.crop((x0, y0, x1, y1))
    a = np.asarray(crop)
    h, w = a.shape[:2]
    side = max(h, w)
    top, left = (side-h)//2, (side-w)//2
    square = np.pad(a, ((top, side-h-top), (left, side-w-left), (0, 0)), mode='edge')
    roi = Image.fromarray(square).resize(
        (CLS_IMAGE_SIZE[1], CLS_IMAGE_SIZE[0]), Image.Resampling.BILINEAR)
    return roi, reason

def make_predicted_inputs(frame, split_name):
    records = []
    for begin in range(0, len(frame), 12):
        rows = list(frame.iloc[begin:begin+12].itertuples(index=False))
        originals = [load_rgb(row.image_path) for row in rows]
        images = np.stack([
            decode_segmentation_image(row.image_path).numpy() for row in rows])
        probabilities = predict_segmentation(
            images, four_view=(SELECTED_SEGMENTATION_METHOD == '4-flip average'))
        for row, original, probability in zip(rows, originals, probabilities):
            binary = probability >= SEG_THRESHOLD
            roi, reason = crop_from_prediction(original, binary)
            mask_path = PRED_MASK_DIR / f'{row.image_id}_pred.png'
            roi_path = ROI_DIR / f'{row.image_id}_roi.jpg'
            Image.fromarray(np.uint8(binary)*255).save(mask_path)
            roi.save(roi_path, quality=95)
            records.append({'image_id': row.image_id, 'roi_path': str(roi_path),
                            'pred_mask_path': str(mask_path), 'roi_reason': reason})
        if (begin + len(rows)) % 120 == 0 or begin + len(rows) == len(frame):
            print(split_name, begin + len(rows), '/', len(frame))
    return frame.drop(columns=['roi_path', 'pred_mask_path', 'roi_reason'],
                      errors='ignore').merge(
        pd.DataFrame(records), on='image_id', validate='one_to_one')

train_df = make_predicted_inputs(train_df, 'train')
val_df = make_predicted_inputs(val_df, 'validation')
test_df = make_predicted_inputs(test_df, 'official test')
for name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    print(name, 'ROI reasons:', frame['roi_reason'].value_counts().to_dict())

examples = val_df.head(4)
fig, axes = plt.subplots(4, 4, figsize=(13, 12))
for i, (_, row) in enumerate(examples.iterrows()):
    original = load_rgb(row['image_path'])
    pred_mask = np.asarray(load_mask(row['pred_mask_path'])) > 127
    expert_mask = np.asarray(load_mask(row['mask_path'])) > 127
    panels = [original, segmentation_overlay(original, expert_mask),
              segmentation_overlay(original, pred_mask), load_rgb(row['roi_path'])]
    for j, (panel, title) in enumerate(zip(panels, ['Original', 'Expert mask',
                                                    'Predicted mask', 'Predicted ROI'])):
        axes[i, j].imshow(panel)
        axes[i, j].set_title(f"{row['image_id']} — {title}" if j == 0 else title)
        axes[i, j].axis('off')
plt.tight_layout(); plt.show()


## 10. Classifier input: original image plus predicted lesion ROI

All variants use the **same** ROI and augmentation. Only the class weights differ.
The classifier never receives an expert mask as input.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_classifier_image(path):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.cast(tf.image.resize(image, CLS_IMAGE_SIZE), tf.float32)

def classifier_dataset(frame, view, training=False):
    ds = tf.data.Dataset.from_tensor_slices((
        frame['image_path'].astype(str).to_numpy(),
        frame['roi_path'].astype(str).to_numpy(),
        frame['label'].astype(np.float32).to_numpy(),
    ))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)

    def convert(full_path, roi_path, label):
        full = decode_classifier_image(full_path)
        if view == 'dual':
            roi = decode_classifier_image(roi_path)
            return {'full_image': full, 'roi_image': roi}, label
        return full, label

    return ds.map(convert, num_parallel_calls=AUTOTUNE).batch(CLS_BATCH_SIZE).prefetch(AUTOTUNE)

from sklearn.utils.class_weight import compute_class_weight
train_labels = train_df['label'].to_numpy()
values = np.unique(train_labels)
weights = compute_class_weight('balanced', classes=values, y=train_labels)
CLASS_WEIGHTS = {int(k): float(v) for k, v in zip(values, weights)}
print('Training class weights:', CLASS_WEIGHTS)


## 11. Full-image + predicted-ROI EfficientNetB0

Both inputs use pixel values 0–255, as required by Keras EfficientNetB0.
The only new training augmentation is modest image brightness variation.


In [ ]:
def augmentation(name, seed):
    return keras.Sequential([
        layers.RandomFlip('horizontal_and_vertical', seed=seed),
        layers.RandomRotation(0.06, seed=seed+1),
        layers.RandomContrast(0.12, seed=seed+2),
        layers.RandomBrightness(0.06, value_range=(0, 255), seed=seed+3),
    ], name=name)

def build_classifier(view, pretrained_weights='imagenet'):
    backbone = keras.applications.EfficientNetB0(
        include_top=False, weights=pretrained_weights, input_shape=(*CLS_IMAGE_SIZE, 3),
        name='efficientnetb0')
    backbone.trainable = False
    gap = layers.GlobalAveragePooling2D(name='gap')
    full_input = keras.Input(shape=(*CLS_IMAGE_SIZE, 3), name='full_image')
    full = augmentation('full_augmentation', SEED)(full_input)
    full_features = gap(backbone(full, training=False))
    if view == 'dual':
        roi_input = keras.Input(shape=(*CLS_IMAGE_SIZE, 3), name='roi_image')
        roi = augmentation('roi_augmentation', SEED+17)(roi_input)
        roi_features = gap(backbone(roi, training=False))
        features = layers.Concatenate(name='combine_features')(
            [full_features, roi_features])
        inputs = [full_input, roi_input]
    else:
        features = full_features
        inputs = full_input
    x = layers.Dense(128, activation='relu', name='hidden_dense')(features)
    x = layers.Dropout(0.35, name='head_dropout')(x)
    result = layers.Dense(1, activation='sigmoid', name='prediction')(x)
    return keras.Model(inputs, result, name=f'melanoma_{view}'), backbone

def compile_classifier(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.AUC(name='auc'),
                 keras.metrics.BinaryAccuracy(name='accuracy')],
    )


## 12. Train one balanced-weight classifier

Save checkpoints using the same classification goal as the final selection:
maximum specificity with at least 80% validation sensitivity; AUC breaks ties.
Save the best frozen and fine-tuned stages. Batch-normalisation stays frozen
while fine-tuning, and the backbone is always called with `training=False`.


In [ ]:
def classification_counts(y_true, probability, threshold):
    probability = np.asarray(probability).reshape(-1)
    y_true = np.asarray(y_true, dtype=int).reshape(-1)
    predicted = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    sens = float(tp / (tp + fn))
    spec = float(tn / (tn + fp))
    return {'Threshold': float(threshold), 'TP': int(tp), 'TN': int(tn),
            'FP': int(fp), 'FN': int(fn), 'Sensitivity': sens,
            'Specificity': spec, 'Accuracy': float((tp + tn) / len(y_true)),
            'Balanced accuracy': (sens + spec) / 2}

def best_operating_point(y_true, probabilities, target_sensitivity):
    thresholds = np.unique(np.r_[0.0, probabilities, 1.0])
    rows = [classification_counts(y_true, probabilities, t) for t in thresholds]
    eligible = [r for r in rows if r['Sensitivity'] >= target_sensitivity]
    if not eligible:
        raise ValueError('No threshold meets target sensitivity.')
    return max(eligible, key=lambda r: (r['Specificity'], r['Accuracy'],
                                        r['Threshold'])), rows

class OperatingPointCheckpoint(keras.callbacks.Callback):
    def __init__(self, dataset, labels, path):
        super().__init__()
        self.dataset, self.labels, self.path = dataset, labels, path
        self.best_key = (-np.inf, -np.inf, -np.inf)
        self.rows = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        probabilities = self.model.predict(self.dataset, verbose=0).ravel()
        operating, _ = best_operating_point(
            self.labels, probabilities, TARGET_VALIDATION_SENSITIVITY)
        auc = float(roc_auc_score(self.labels, probabilities))
        logs['val_specificity_at_target'] = operating['Specificity']
        key = (operating['Specificity'], auc, operating['Accuracy'])
        self.rows.append({'epoch': epoch+1, 'AUC': auc, **operating})
        if key > self.best_key:
            self.best_key = key
            self.model.save(str(self.path))
        print(f" val specificity@target={operating['Specificity']:.4f}; "
              f"sensitivity={operating['Sensitivity']:.4f}; AUC={auc:.4f}")

def fit_classifier():
    keras.utils.set_random_seed(SEED)
    train_ds = classifier_dataset(train_df, 'dual', training=True)
    validation_ds = classifier_dataset(val_df, 'dual')
    labels = val_df.label.astype(int).to_numpy()
    model, _ = build_classifier('dual')
    head_path = MODELS_DIR / 'dual_balanced_brightness_head.keras'
    fine_path = MODELS_DIR / 'dual_balanced_brightness_fine.keras'
    checkpoint = OperatingPointCheckpoint(validation_ds, labels, head_path)
    compile_classifier(model, CLS_HEAD_LR)
    model.fit(train_ds, validation_data=validation_ds, epochs=CLS_HEAD_EPOCHS,
              class_weight=CLASS_WEIGHTS,
              callbacks=[checkpoint, keras.callbacks.EarlyStopping(
                  monitor='val_specificity_at_target', mode='max', patience=5)],
              verbose=1)
    pd.DataFrame(checkpoint.rows).to_csv(RESULTS_DIR / 'head_epoch_metrics.csv', index=False)
    del model
    gc.collect()
    keras.backend.clear_session()

    model = keras.models.load_model(str(head_path))
    backbone = model.get_layer('efficientnetb0')
    backbone.trainable = True
    for layer in backbone.layers[:-20]:
        layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    compile_classifier(model, CLS_FINE_TUNE_LR)
    checkpoint = OperatingPointCheckpoint(validation_ds, labels, fine_path)
    model.fit(train_ds, validation_data=validation_ds, epochs=CLS_FINE_TUNE_EPOCHS,
              class_weight=CLASS_WEIGHTS,
              callbacks=[checkpoint, keras.callbacks.EarlyStopping(
                  monitor='val_specificity_at_target', mode='max', patience=4)],
              verbose=1)
    pd.DataFrame(checkpoint.rows).to_csv(RESULTS_DIR / 'fine_epoch_metrics.csv', index=False)
    del model
    gc.collect()
    keras.backend.clear_session()
    return [{'Stage': stage, 'Model path': str(path)} for stage, path in
            [('frozen head', head_path), ('fine-tuned', fine_path)]]

print('Training with class weights:', CLASS_WEIGHTS)
candidate_models = fit_classifier()


## 13. Select checkpoint, prediction method, and threshold on validation

Four-view test-time augmentation (TTA) averages predictions for the original,
horizontal flip, vertical flip and both flips. Both the full image and ROI
receive the same flip. Select whichever checkpoint/method has the highest
validation specificity at **at least 80% validation sensitivity**; break ties
by AUC. No official-test data is used for this selection.


In [ ]:
def predict_with_flips(model, frame, four_view=False):
    # classifier_dataset does not shuffle evaluation rows; the predictions
    # retain the same order as frame. Augmentation layers are inactive at predict.
    base_ds = classifier_dataset(frame, 'dual', training=False)
    if not four_view:
        return model.predict(base_ds, verbose=0).ravel()
    predictions = []
    for flip_lr, flip_ud in [(False, False), (True, False),
                             (False, True), (True, True)]:
        def apply_flip(features, target, flip_lr=flip_lr, flip_ud=flip_ud):
            views = {}
            for key, pixels in features.items():
                if flip_lr:
                    pixels = tf.image.flip_left_right(pixels)
                if flip_ud:
                    pixels = tf.image.flip_up_down(pixels)
                views[key] = pixels
            return views, target
        flip_ds = base_ds.map(apply_flip, num_parallel_calls=AUTOTUNE)
        predictions.append(model.predict(flip_ds, verbose=0).ravel())
    return np.mean(np.stack(predictions), axis=0)

y_val = val_df['label'].astype(int).to_numpy()
selection_rows = []
all_threshold_rows = []
candidate_probabilities = {}
for candidate in candidate_models:
    model = keras.models.load_model(candidate['Model path'])
    for method, four_view in [('single', False), ('4-flip average', True)]:
        probabilities = predict_with_flips(model, val_df, four_view=four_view)
        key = (candidate['Stage'], method)
        candidate_probabilities[key] = probabilities
        auc = float(roc_auc_score(y_val, probabilities))
        best, rows = best_operating_point(
            y_val, probabilities, TARGET_VALIDATION_SENSITIVITY)
        for row in rows:
            all_threshold_rows.append({'Stage': candidate['Stage'],
                                       'Method': method,
                                       'Validation AUC': auc, **row})
        selection_rows.append({'Stage': candidate['Stage'], 'Method': method,
                               'Model path': candidate['Model path'],
                               'Validation AUC': auc, **best})
    del model
    gc.collect()

selection_table = pd.DataFrame(selection_rows).sort_values(
    ['Specificity', 'Validation AUC', 'Accuracy'], ascending=False
).reset_index(drop=True)
display(selection_table.drop(columns='Model path').round(4))
winner = selection_table.iloc[0]
SELECTED_STAGE = str(winner['Stage'])
SELECTED_METHOD = str(winner['Method'])
SELECTED_THRESHOLD = float(winner['Threshold'])
SELECTED_VIEW = 'dual'
selected_classifier = keras.models.load_model(winner['Model path'])
print('Selected on validation:', SELECTED_STAGE, SELECTED_METHOD,
      'threshold', SELECTED_THRESHOLD)

selection_table.to_csv(RESULTS_DIR / 'validation_candidate_comparison.csv', index=False)
pd.DataFrame(all_threshold_rows).to_csv(
    RESULTS_DIR / 'validation_all_thresholds.csv', index=False)
selected_val_prob = candidate_probabilities[(SELECTED_STAGE, SELECTED_METHOD)]
val_predictions = val_df[['image_id', 'diagnosis', 'label', 'image_path',
                          'roi_path', 'pred_mask_path']].copy()
val_predictions['malignancy_probability'] = selected_val_prob
val_predictions['predicted_label'] = (
    selected_val_prob >= SELECTED_THRESHOLD).astype(int)
val_predictions['outcome'] = [
    ('TP' if truth == 1 and pred == 1 else
     'TN' if truth == 0 and pred == 0 else
     'FP' if truth == 0 and pred == 1 else 'FN')
    for truth, pred in zip(y_val, val_predictions['predicted_label'])
]
val_predictions.to_csv(RESULTS_DIR / 'selected_validation_predictions.csv',
                       index=False)
selected_model_path = MODELS_DIR / 'selected_final_classifier.keras'
selected_classifier.save(str(selected_model_path))
(RESULTS_DIR / 'selected_configuration.json').write_text(json.dumps({
    'view': SELECTED_VIEW, 'class_weights': CLASS_WEIGHTS,
    'stage': SELECTED_STAGE, 'method': SELECTED_METHOD,
    'threshold': SELECTED_THRESHOLD,
    'target_validation_sensitivity': TARGET_VALIDATION_SENSITIVITY,
    'segmenter_reused': SEGMENTER_REUSED,
    'segmentation_method': SELECTED_SEGMENTATION_METHOD,
    'segmentation_threshold': SEG_THRESHOLD,
    'roi_padding': 0.15, 'split_sha256': split_hash,
    'note': 'Official test previously viewed; final follow-up is exploratory.',
}, indent=2))


## 14. Exploratory official-test evaluation

The checkpoint, prediction method and threshold are locked from validation.
This official test was viewed in prior runs, so report it as exploratory.


In [ ]:
official_test_ds = classifier_dataset(test_df, SELECTED_VIEW)
y_test = test_df['label'].astype(int).to_numpy()
test_prob = predict_with_flips(
    selected_classifier, test_df, four_view=(SELECTED_METHOD == '4-flip average'))
test_metrics = classification_counts(y_test, test_prob, SELECTED_THRESHOLD)
test_metrics['ROC-AUC'] = float(roc_auc_score(y_test, test_prob))
test_metrics['F1'] = float(f1_score(y_test,
                                  (test_prob >= SELECTED_THRESHOLD).astype(int)))
test_metrics['Input'] = SELECTED_VIEW
test_metrics['Method'] = SELECTED_METHOD
test_metrics['Class weights'] = 'balanced'
test_metrics['Stage'] = SELECTED_STAGE
test_metrics['N'] = len(test_df)
official_test_metrics = pd.DataFrame([test_metrics])
display(official_test_metrics.T.rename(columns={0: 'Exploratory follow-up result'}))
official_test_metrics.to_csv(RESULTS_DIR / 'FOLLOWUP_official_test_classification_metrics.csv',
                             index=False)
test_predictions = test_df[['image_id', 'diagnosis', 'label', 'image_path',
                            'roi_path', 'pred_mask_path']].copy()
test_predictions['malignancy_probability'] = test_prob
test_predictions['predicted_label'] = (test_prob >= SELECTED_THRESHOLD).astype(int)
test_predictions['outcome'] = [
    ('TP' if truth == 1 and pred == 1 else
     'TN' if truth == 0 and pred == 0 else
     'FP' if truth == 0 and pred == 1 else 'FN')
    for truth, pred in zip(y_test, test_predictions['predicted_label'])]
test_predictions.to_csv(RESULTS_DIR / 'FOLLOWUP_official_test_predictions.csv', index=False)

matrix = confusion_matrix(y_test, test_predictions['predicted_label'], labels=[0, 1])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(matrix, cmap='Blues')
axes[0].set_xticks([0, 1], labels=['Benign', 'Melanoma'])
axes[0].set_yticks([0, 1], labels=['Benign', 'Melanoma'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Exploratory follow-up confusion matrix')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(matrix[i, j]), ha='center', va='center', fontsize=16)
fpr, tpr, _ = roc_curve(y_test, test_prob)
axes[1].plot(fpr, tpr, label=f"AUC = {test_metrics['ROC-AUC']:.3f}")
axes[1].plot([0, 1], [0, 1], '--', color='grey')
axes[1].set_xlabel('False positive rate'); axes[1].set_ylabel('Sensitivity')
axes[1].set_title('Exploratory follow-up ROC'); axes[1].legend()
plt.tight_layout(); plt.show()


## 15. Segmentation metrics on the official test set

In [ ]:
segment_rows = []
for row in test_df.itertuples(index=False):
    truth = decode_segmentation_mask(row.mask_path).numpy()[..., 0] > 0.5
    prediction = np.asarray(load_mask(row.pred_mask_path)) > 127
    intersection = np.logical_and(truth, prediction).sum()
    union = np.logical_or(truth, prediction).sum()
    dice = (2*intersection+1e-7)/(truth.sum()+prediction.sum()+1e-7)
    iou = (intersection+1e-7)/(union+1e-7)
    segment_rows.append({'image_id': row.image_id, 'dice': dice, 'iou': iou})
segmentation_test = pd.DataFrame(segment_rows)
print('Official test Dice (per-image mean):', segmentation_test['dice'].mean())
print('Official test IoU (per-image mean):', segmentation_test['iou'].mean())
print('Median / worst Dice:', segmentation_test['dice'].median(),
      segmentation_test['dice'].min())
segmentation_test.to_csv(RESULTS_DIR / 'official_test_segmentation_per_image.csv',
                         index=False)

# Check the worst failures rather than relying on mean Dice alone.
for row in segmentation_test.nsmallest(3, 'dice').itertuples(index=False):
    record = test_df.loc[test_df.image_id == row.image_id].iloc[0]
    original = load_rgb(record.image_path)
    expert = np.asarray(load_mask(record.mask_path).resize(
        (original.width, original.height), Image.Resampling.NEAREST)) > 127
    pred = np.asarray(load_mask(record.pred_mask_path).resize(
        (original.width, original.height), Image.Resampling.NEAREST)) > 127
    fig, axes = plt.subplots(1, 4, figsize=(13, 3))
    for ax, panel, title in zip(axes, [original,
            segmentation_overlay(original, expert),
            segmentation_overlay(original, pred), load_rgb(record.roi_path)],
            ['Original', 'Expert mask', 'Predicted mask', 'Predicted ROI']):
        ax.imshow(panel)
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f'{row.image_id}: Dice={row.dice:.3f}')
    plt.tight_layout()
    plt.show()


## 16. Show representative lesion masks, ROIs, and Grad-CAM

Grad-CAM visualises the selected model's spatial evidence for a prediction;
it does not establish a clinical reason. For the dual-view arm, the heatmap
shown here is for the ROI branch.

In [ ]:
def make_gradcam_heatmap(model, full_batch, roi_batch, view, target_class=1):
    backbone = model.get_layer('efficientnetb0')
    gap = model.get_layer('gap')
    with tf.GradientTape() as tape:
        full_map = backbone(full_batch, training=False)
        if view == 'dual':
            roi_map = backbone(roi_batch, training=False)
            feature_map = roi_map
            # Watch before constructing downstream pooling/head operations.
            # This also works when all backbone weights are frozen.
            tape.watch(feature_map)
            features = model.get_layer('combine_features')(
                [gap(full_map), gap(feature_map)])
        else:
            feature_map = full_map
            tape.watch(feature_map)
            features = gap(feature_map)
        x = model.get_layer('hidden_dense')(features)
        x = model.get_layer('head_dropout')(x, training=False)
        prediction_layer = model.get_layer('prediction')
        logit = tf.linalg.matmul(x, prediction_layer.kernel) + prediction_layer.bias
        score = logit[:, 0] if target_class == 1 else -logit[:, 0]
    gradients = tape.gradient(score, feature_map)
    if gradients is None:
        raise RuntimeError('Grad-CAM gradient unavailable for selected view.')
    pooled = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)
    heatmap = tf.nn.relu(tf.reduce_sum(feature_map*pooled, axis=-1)[0])
    return (heatmap/(tf.reduce_max(heatmap)+1e-8)).numpy()

def heatmap_overlay(pil, heatmap):
    image = pil.convert('RGB').resize((CLS_IMAGE_SIZE[1], CLS_IMAGE_SIZE[0]))
    heat = Image.fromarray(np.uint8(np.clip(heatmap, 0, 1)*255)).resize(
        image.size, Image.Resampling.BILINEAR)
    colors = plt.get_cmap('jet')(np.asarray(heat, dtype=np.float32)/255.0)[..., :3]
    base = np.asarray(image, dtype=np.float32)/255.0
    return Image.fromarray(np.uint8(255*np.clip(0.62*base+0.38*colors, 0, 1)))

representatives = []
for outcome in ('TP', 'TN', 'FP', 'FN'):
    matching = test_predictions.loc[test_predictions['outcome'] == outcome]
    if len(matching):
        representatives.append(matching.iloc[0])
for row in representatives:
    full = load_rgb(row['image_path'])
    roi = load_rgb(row['roi_path'])
    full_input = decode_classifier_image(row['image_path'])[None, ...]
    roi_input = decode_classifier_image(row['roi_path'])[None, ...]
    heatmap = make_gradcam_heatmap(
        selected_classifier, tf.convert_to_tensor(full_input),
        tf.convert_to_tensor(roi_input), SELECTED_VIEW,
        target_class=int(row['predicted_label']))
    image_for_heatmap = roi if SELECTED_VIEW == 'dual' else full
    predicted_mask = np.asarray(load_mask(row['pred_mask_path'])) > 127
    panels = [full, segmentation_overlay(full, predicted_mask), roi,
              heatmap_overlay(image_for_heatmap, heatmap)]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, panel, title in zip(
        axes, panels,
        ['Original', 'Predicted lesion mask', 'Lesion ROI',
         'Single-view Grad-CAM on ROI']):
        ax.imshow(panel); ax.set_title(title); ax.axis('off')
    fig.suptitle(f"{row['image_id']} — {row['outcome']} — "
                 f"Melanoma score={row['malignancy_probability']:.3f} ({SELECTED_METHOD})")
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / f"gradcam_{row['image_id']}_{row['outcome']}.png",
                dpi=130, bbox_inches='tight')
    plt.show()
if SAVE_RESULTS_TO_DRIVE and OUTPUT_DIR != RUNTIME_ROOT / 'results':
    shutil.make_archive(str(OUTPUT_DIR / 'predicted_masks'), 'zip', PRED_MASK_DIR)
    shutil.make_archive(str(OUTPUT_DIR / 'lesion_rois'), 'zip', ROI_DIR)
print('Selected model:', selected_model_path)
print('Tables:', RESULTS_DIR)


## Report this candidate to your team

- Share the selection table, exploratory 379-image test metrics, TP/TN/FP/FN,
  and per-image segmentation CSV. Keep the exact split IDs and threshold.
- Compare pipelines under the same training/validation split, labels, target
  sensitivity and evaluation definitions. Multiple model runs on the already
  viewed test set do not provide a fresh independent final estimate.
- The model is a research prototype; do not describe its predictions as an
  accurate or clinically validated melanoma diagnosis.
